In [ ]:
import os
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

load_dotenv()

model = init_chat_model("gemini-3.6-flash", model_provider="google_genai")

### Messages in LangChain

Messages are the fundamental building blocks for representing conversations between users, AI models, tools, and the system.

Modern chat models (such as Gemini, Claude, and GPT-4) work with structured lists of messages (List[BaseMessage]) rather than a single raw text string.

---

### Core Attributes of a Message Object

Every message object in LangChain contains:
* Type / Role: Identifies the message category (type in LangChain, role in LLM APIs like system, user, assistant, tool).
* Content: The actual payload (string text or a list of multimodal dicts like images/audio).
* Metadata: Additional fields such as response_metadata, usage_metadata (tokens), tool_calls, and additional_kwargs.

### Message Types & Roles

Below is the complete reference of all message types and roles available in LangChain:

| Class Name | LangChain .type | LLM API role | Purpose & Description |
| :--- | :--- | :--- | :--- |
| SystemMessage | system | system / developer | Sets instructions, persona, and behavioral rules for the LLM. |
| HumanMessage | human | user | Represents user input, prompts, and questions. |
| AIMessage | ai | assistant / model | Represents responses generated by the model (text or tool calls). |
| ToolMessage | tool | tool | Output returned from executing an external tool or function. |
| ChatMessage | chat | Custom (role=...) | Arbitrary role string for non-standard providers or agent roles. |


### 1. SystemMessage

A SystemMessage provides high-priority context, guidelines, and behavioral boundaries to the LLM before user interaction starts.


In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage

system_msg = SystemMessage("""
You are a senior Python developer with expertise in web frameworks.
Always provide code examples and explain your reasoning.
Be concise but thorough in your explanations.
""")

messages = [
    system_msg,
    HumanMessage(content="How do I create a REST API?")
]
response = model.invoke(messages)

### 2. HumanMessage

A HumanMessage represents user inputs and interactions. It can hold plain text or multimodal content (images, audio, files).

You can also attach metadata like user_id or session_id using additional_kwargs.


In [ ]:
from langchain_core.messages import HumanMessage

human_message = HumanMessage(
    content="""
I am a Java Backend Developer with 3 years of experience preparing for SDE-2 interviews.
Explain how Messages work in LangChain step by step with Python code examples.
""",
    additional_kwargs={
        "user_id": "12345",
        "session_id": "session_001",
        "source": "web_app"
    }
)

print(f"Type: {human_message.type}")
print(f"Content preview: {human_message.content[:50]}...")

In [ ]:
response = model.invoke([human_message])
print(response.content)

### 3. AIMessage

An AIMessage represents responses generated by the LLM after invoking the model.

* Request: HumanMessage (role="user")
* Response: AIMessage (role="assistant")

An AIMessage contains:
* content: Text response generated by the model.
* tool_calls: Requested tool/function calls (if the model decides to use a tool).
* usage_metadata: Token counts (input_tokens, output_tokens, total_tokens).
* response_metadata: Model details (finish reason, safety ratings, response headers).


In [ ]:
print(f"AIMessage type: {response.type}")
print(f"Usage metadata: {response.usage_metadata}")

### 4. Conversation History

LLMs are stateless. To maintain context across chat turns, you must pass the entire history of [HumanMessage, AIMessage, HumanMessage, ...] with each request.


In [ ]:
messages = [
    HumanMessage(content="What is Java?")
]

for _ in range(2):
    response = model.invoke(messages)

    print("AI:", response.content[:100], "...")
    print("-" * 50)

    # Append AI response to conversation history
    messages.append(response)

    # Append next user question
    messages.append(HumanMessage(content="Who developed Java?"))

### 5. ToolMessage

A ToolMessage represents the output returned after executing an external tool or function (e.g., database lookup, API request, search query).

When an AIMessage requests tool execution via tool_calls, your backend executes the tool and passes the output back as a ToolMessage linked by tool_call_id.


In [ ]:
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage

# 1. User question
messages = [HumanMessage(content="What's the weather in Delhi?")]

# 2. AI requests tool execution
ai_message = AIMessage(
    content="",
    tool_calls=[
        {
            "name": "get_weather",
            "args": {"city": "Delhi"},
            "id": "call_001"
        }
    ]
)
messages.append(ai_message)

# 3. Execute tool in backend
tool_result = "Temperature: 32°C, Sunny"

# 4. Create ToolMessage with matching tool_call_id
tool_message = ToolMessage(content=tool_result, tool_call_id="call_001")
messages.append(tool_message)

# 5. Final AI response
final_ai = AIMessage(content="The current weather in Delhi is 32°C and sunny.")
messages.append(final_ai)

print("Final Conversation Stack:")
for msg in messages:
    print(f"[{msg.type.upper()}] (class: {msg.__class__.__name__}): {msg.content or msg.tool_calls}")

### 6. ChatMessage

When integrating with models or custom backends that support non-standard roles (e.g., role="moderator", role="admin", role="instruction"), use ChatMessage(role=...).


NOTE : Chatmessage is not more used now, since we already have multiple message type, u can use it for custom roles

In [ ]:
from langchain_core.messages import ChatMessage

custom_msg = ChatMessage(
    role="moderator",
    content="Notice: Please keep discussion relevant to programming."
)

print(f"Type: {custom_msg.type}")
print(f"Custom Role: {custom_msg.role}")
print(f"Content: {custom_msg.content}")

### 7. Multimodal Messages

Modern chat models accept multimodal inputs. In LangChain, you pass a list of content blocks (dicts) as the content of a HumanMessage.


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage

# Direct Gemini Chat Model instantiation
llm = ChatGoogleGenerativeAI(model="gemini-3.6-flash")

message = HumanMessage(
    content=[
        {
            "type": "text",
            "text": "Describe the content of this image in detail."
        },
        {
            "type": "image_url",
            "image_url": {
                "url": "https://cdn.pixabay.com/photo/2015/12/30/21/34/labrador-1114810_1280.jpg"
            }
        }
    ]
)

response = llm.invoke([message])
print(response.content)

### 8. Message Trimming

Long conversations can exceed an LLM's maximum token context limit. trim_messages lets you automatically trim history down to a specified token or message budget while retaining the SystemMessage.


In [ ]:
from langchain_core.messages import (
    SystemMessage, HumanMessage, AIMessage, trim_messages
)

conversation = [
    SystemMessage(content="You are a helpful customer support assistant."),
    HumanMessage(content="Hi, I need help with my order."),
    AIMessage(content="Hello! I would be happy to help. What is your order number?"),
    HumanMessage(content="My order number is #98765."),
    AIMessage(content="Thank you. I see your order #98765 is out for delivery today!"),
    HumanMessage(content="Great, what time will it arrive?")
]

# Trim history to keep system message and last N messages
trimmed = trim_messages(
    conversation,
    max_tokens=4,
    strategy="last",
    token_counter=len, # Count by message objects or pass custom token counter
    include_system=True,
    start_on="human"
)

print(f"Original message count: {len(conversation)}")
print(f"Trimmed message count: {len(trimmed)}")
for m in trimmed:
    print(f"- [{m.type.upper()}] (class: {m.__class__.__name__}): {m.content}")

### 9. Message Serialization

When building web applications, you often need to store conversation history in a database or Redis (as JSON) and reconstruct LangChain message objects when loading user sessions.


In [ ]:
from langchain_core.messages import (
    messages_to_dict, messages_from_dict, SystemMessage, HumanMessage, AIMessage
)

history = [
    SystemMessage(content="You are a coding assistant."),
    HumanMessage(content="How do I sort a list in Python?"),
    AIMessage(content="Use the sorted() function or list.sort() method.")
]

# 1. Convert LangChain message objects to plain Python dictionaries (JSON serializable)
serialized_history = messages_to_dict(history)
print("Serialized JSON-ready Dicts:")
print(serialized_history[0])

# 2. Reconstruct LangChain message objects from dicts
restored_history = messages_from_dict(serialized_history)
print(f"\nRestored {len(restored_history)} message objects successfully!")